# Object detection data preparation

In this notebook, we prepare the data for the object detection with YOLO and Ultralytics. 

Steps of notebook:
* Create folders for data
* Copy Sentinel-2 1C data to own folder from the course pre-downloaded data
* Downlaod training annotations
* Convert the data into YOLO compatible form, using [geoml2](https://github.com/mayrajeo/geo2ml).

In [ ]:
import os
from pathlib import Path

# Data download from URL
import requests

# YOLO format training data creation
from geo2ml.scripts.data import create_yolo_dataset

Set folders.

In [ ]:
exercise_folder = os.path.join(os.sep, 'scratch', 'project_462001167', 'students', os.environ.get('USER'), 'GeoML', '09_object_detection', 'own_model_training') 
sentinel_folder = os.path.join(exercise_folder, 'sentinel2')
annotations_folder = os.path.join(exercise_folder, 'annotations')
yolo_data_folder = os.path.join(exercise_folder, 'yolo_data')

Define URLs in Zenodo for labels data. Retrieved from https://doi.org/10.5281/zenodo.15019034.

In [ ]:
urls = {
    "34VEM.gpkg": "https://zenodo.org/records/15019034/files/34VEM.gpkg?download=1",
    "35VLG.gpkg": "https://zenodo.org/records/15019034/files/35VLG.gpkg?download=1",
    "34VEN.gpkg": "https://zenodo.org/records/15019034/files/34VEN.gpkg?download=1"
}

Make sure that you are in the right folder. Add new folders for the labels and prepared data for YOLO model.

In [ ]:
os.chdir(exercise_folder)

if not os.path.isdir(sentinel_folder):
    os.makedirs(sentinel_folder)

if not os.path.isdir(annotations_folder):
    os.makedirs(annotations_folder)

if not os.path.isdir(yolo_data_folder):
    os.makedirs(yolo_data_folder)

During the course, copy pre-downloaded images to your own folder.

In [ ]:
! cp -r /scratch/project_462001167/09_sentinel_images/*.tif {sentinel_folder}

Download annotations from Zenodo.

In [ ]:
import urllib.request
for filename, url in urls.items():
    path = os.path.join(annotations_folder, filename)
    urllib.request.urlretrieve(url, path)

Define split of images between training, validation and test datasets.

In [ ]:
datasets = {'34VEM' : 'train', '35VLG' : 'val', '34VEN': 'test'}

Create YOLO compatible datasets using [geo2ml](https://mayrajeo.github.io/geo2ml/)-library.

It creates folders train, val and test. All of these folders contain sub-folder:
* `images` - the tiled raster patches.
* `vectors`- GeoPackge-files corresponding to each file in images, if the location contains any annotations.
* `labels` - the annotations in YOLO format.
* file `yolo.yaml`, which can be used as a template for the dataset description file

This gives warning `Specify layer parameter to avoid this warning.`, but that can be ignored.

In [ ]:
sentinel_folder_path = Path(sentinel_folder)
for raster in sentinel_folder_path.glob('*.tif'): 
    name = raster.name.split("_")
    layer = name[1][:8]
    annotation = name[0][1:]
    print(str(raster.name) + annotation + ' ' + layer)
    print("adding", raster.name, "to", datasets[annotation])
    print(layer)
    
    raster_path = os.path.join(sentinel_folder, raster)
    polygon_file = annotation + ".gpkg"
    poly_path = os.path.join(annotations_folder, polygon_file)
    out_folder = os.path.join(yolo_data_folder, datasets[annotation])
    out_folder_path = Path(out_folder)


    #!ogrinfo {poly_path} {layer} -so
    create_yolo_dataset(raster_path=raster_path, polygon_path=poly_path, target_column='id',
                    gpkg_layer=layer, outpath=out_folder_path, output_format='gpkg', save_grid=False,
                    gridsize_x=320, gridsize_y=320, ann_format='box', min_bbox_area=0)

*Please be patient, this takes ~10-15 minutes.*

*It prints some erros, but these are ok.*